# Zero-Shot SBIR: Kaggle GPU Training & Cross-Dataset Evaluation Pipeline
**Architecture:** Asymmetric Hybrid LoRA-CLIP (ViT-B/32)
**Training Dataset:** Sketchy Database (Split 2: 104 Seen Classes)
**Testing Datasets:**
1. Sketchy Unseen (21 Classes) - In-Domain Zero-Shot Benchmark
2. TU-Berlin Extended (30 Unseen Classes or 250 Classes) - Cross-Dataset Generalization Benchmark

## Step 1: Clone Repository & Install Dependencies
Make sure **GPU T4 x2** or **P100** is selected in Kaggle Notebook settings (Settings -> Accelerator -> GPU).

In [ ]:
import os

# Clone if not present, otherwise pull latest code
if not os.path.exists("/kaggle/working/ZS-SBIR-LoRA-CLIP"):
    !git clone https://github.com/Quandinh030106/ZS-SBIR-LoRA-CLIP.git /kaggle/working/ZS-SBIR-LoRA-CLIP

%cd /kaggle/working/ZS-SBIR-LoRA-CLIP
!git pull origin main
!pip install -q -r requirements.txt
!python train.py --self-test

## Step 2: Configure & Verify Dataset Paths
Directly uses the exact paths for **Sketchy Images Dataset** and **TUBerlin** under `/kaggle/input/`.

In [ ]:
import os

# Exact Kaggle paths for attached datasets
SKETCHY_ROOT = "/kaggle/input/datasets/dhananjayapaliwal/fulldataset/temp_extraction/256x256"
TUBERLIN_ROOT = "/kaggle/input/datasets/rishikashili/tuberlin/TUBerlin"

# Fallback candidate checks if paths differ on other Kaggle mount points
if not os.path.exists(SKETCHY_ROOT):
    for c in [
        "/kaggle/input/fulldataset/temp_extraction/256x256",
        "/kaggle/input/sketchy-dataset/256x256",
    ]:
        if os.path.exists(c):
            SKETCHY_ROOT = c
            break
    else:
        for root, dirs, _ in os.walk("/kaggle/input"):
            dl = [d.lower() for d in dirs]
            if "photo" in dl and ("splitted_sketches" in dl or "sketch" in dl):
                SKETCHY_ROOT = root
                break

if not os.path.exists(TUBERLIN_ROOT):
    for c in [
        "/kaggle/input/tuberlin/TUBerlin",
        "/kaggle/input/tuberlin",
    ]:
        if os.path.exists(c):
            TUBERLIN_ROOT = c
            break
    else:
        for root, dirs, _ in os.walk("/kaggle/input"):
            dl = [d.lower() for d in dirs]
            if any("imageresized" in d for d in dl) and any("png" in d for d in dl):
                TUBERLIN_ROOT = root
                break

print("=" * 65)
print(f"SKETCHY_ROOT:  {SKETCHY_ROOT}  [Found: {os.path.exists(str(SKETCHY_ROOT))}]")
print(f"TUBERLIN_ROOT: {TUBERLIN_ROOT}  [Found: {os.path.exists(str(TUBERLIN_ROOT))}]")
print("=" * 65)

assert os.path.exists(str(SKETCHY_ROOT)), f"[ERROR] Sketchy dataset not found at {SKETCHY_ROOT}"

# Export environment variables
os.environ["SKETCHY_ROOT"] = SKETCHY_ROOT
if TUBERLIN_ROOT and os.path.exists(TUBERLIN_ROOT):
    os.environ["TUBERLIN_ROOT"] = TUBERLIN_ROOT

## Step 3: Evaluate Baseline 0 (Frozen CLIP ViT-B/32 on Sketchy Split 2)
Measures pure zero-shot performance before fine-tuning to establish the baseline comparison.

In [ ]:
!python train.py \
  --data-root {SKETCHY_ROOT} \
  --split split2 \
  --eval-baseline-only

## Step 4: Pre-compute Offline Photo Features (Cache)
Pre-computing 512D embeddings for all 10,400 seen photos eliminates disk I/O bottlenecks and accelerates training by 4-6x.

In [ ]:
CACHE_PATH = "/kaggle/working/photo_features_cache.pt"

!python scripts/cache_photo_features.py \
  --data-root {SKETCHY_ROOT} \
  --split split2 \
  --output {CACHE_PATH}

## Step 5: Train Asymmetric LoRA-CLIP Model (15 Epochs)
Trains with In-Batch Semi-Hard Mining, Partial Sketch Simulation (CVPR 2020), Noise-Tolerance (CVPR 2022), and Photo Caching.

In [ ]:
!python train.py \
  --data-root {SKETCHY_ROOT} \
  --split split2 \
  --epochs 15 \
  --batch-size 64 \
  --lr 2e-5 \
  --lora-rank 8 \
  --mining-mode semi_hard \
  --lambda-cls 0.3 \
  --lambda-consist 1.0 \
  --use-cached-photos

## Step 6: Cross-Dataset Generalization Test on TU-Berlin
Evaluates the best checkpoint (trained strictly on Sketchy) directly on TU-Berlin Extended to prove domain generalization without overfitting.

In [ ]:
BEST_CKPT = "outputs/checkpoints/best_seco_lora_clip.pt"

if TUBERLIN_ROOT and os.path.exists(TUBERLIN_ROOT):
    !python scripts/evaluate.py \
      --checkpoint {BEST_CKPT} \
      --data-root {TUBERLIN_ROOT} \
      --dataset tuberlin \
      --split tuberlin \
      --photo-subdir ImageResized_ready \
      --sketch-subdir png_ready \
      --viz-samples 10 \
      --output-dir "/kaggle/working/tuberlin_eval_results"
else:
    print("[NOTE] TU-Berlin path not found. Attach TU-Berlin dataset to Kaggle notebook to run cross-eval.")

## Step 7: Checkpoint Verification & Ready for Web Demo
Download `best_seco_lora_clip.pt` from the Kaggle Output file browser on the right panel.

In [ ]:
ckpt = "outputs/checkpoints/best_seco_lora_clip.pt"
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / (1024 * 1024)
    print(f"SUCCESS: Checkpoint ready ({size_mb:.2f} MB) at: {ckpt}")
    print("Download this file to your local computer to power the Gradio Web Demo!")
else:
    print("Checkpoint not yet generated.")